In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, desc, sum

spark = SparkSession.builder.appName("DMartRetailAnalytics").getOrCreate()

In [3]:
df = spark.read.csv("dmart_sales(in).csv", header=True, inferSchema=True)
df.show(5)

+--------+----------+-----------+--------+-------+---------------+
|store_id|product_id|customer_id|quantity|  price|      timestamp|
+--------+----------+-----------+--------+-------+---------------+
|    S026|     P0561|     C28234|       9| 1947.9| 4/1/2026 14:18|
|    S019|     P0718|     C08674|       2|1101.07| 3/20/2026 2:41|
|    S012|     P0797|     C45415|       4| 100.94|3/30/2026 22:51|
|    S016|     P0446|     C17642|       7|1828.09| 3/28/2026 7:26|
|    S040|     P0005|     C01498|       7|1735.48|3/19/2026 12:18|
+--------+----------+-----------+--------+-------+---------------+
only showing top 5 rows


In [4]:
df_cleaned = df.dropna()
df_cleaned = df_cleaned.filter((col("quantity") > 0) & (col("price") > 0))
df_cleaned = df_cleaned.withColumn("total_value", col("quantity") * col("price"))
df_cleaned = df_cleaned.withColumn("date", to_date(col("timestamp"), "M/d/yyyy H:mm"))
df_cleaned.show(5)

+--------+----------+-----------+--------+-------+---------------+------------------+----------+
|store_id|product_id|customer_id|quantity|  price|      timestamp|       total_value|      date|
+--------+----------+-----------+--------+-------+---------------+------------------+----------+
|    S026|     P0561|     C28234|       9| 1947.9| 4/1/2026 14:18|17531.100000000002|2026-04-01|
|    S019|     P0718|     C08674|       2|1101.07| 3/20/2026 2:41|           2202.14|2026-03-20|
|    S012|     P0797|     C45415|       4| 100.94|3/30/2026 22:51|            403.76|2026-03-30|
|    S016|     P0446|     C17642|       7|1828.09| 3/28/2026 7:26|          12796.63|2026-03-28|
|    S040|     P0005|     C01498|       7|1735.48|3/19/2026 12:18|          12148.36|2026-03-19|
+--------+----------+-----------+--------+-------+---------------+------------------+----------+
only showing top 5 rows


In [5]:
sales_per_store_per_day = df_cleaned.groupBy("store_id", "date").agg(sum("total_value").alias("total_sales"))
sales_per_store_per_day.show()

+--------+----------+------------------+
|store_id|      date|       total_sales|
+--------+----------+------------------+
|    S009|2026-03-09|         3817531.9|
|    S032|2026-03-14|3875893.5100000002|
|    S033|2026-03-12|3710183.9200000004|
|    S031|2026-03-11|3812412.1300000004|
|    S031|2026-03-16|3848062.2899999996|
|    S043|2026-03-12|        4119570.06|
|    S021|2026-03-15|3514283.0000000005|
|    S043|2026-03-07|        3334591.48|
|    S043|2026-03-16|3796301.8200000003|
|    S028|2026-04-03|         490172.16|
|    S022|2026-03-08|        4074496.55|
|    S005|2026-03-19|        3697788.13|
|    S019|2026-03-21|3634687.6400000006|
|    S001|2026-03-23|3381885.7800000003|
|    S038|2026-03-04|3167357.6800000006|
|    S029|2026-03-21|        3846533.29|
|    S034|2026-04-03|         489748.05|
|    S003|2026-03-28|3679466.7399999998|
|    S015|2026-03-21|3480435.0600000005|
|    S023|2026-03-20|3812980.4499999997|
+--------+----------+------------------+
only showing top

In [6]:
top_5_products = df_cleaned.groupBy("product_id").agg(sum("total_value").alias("revenue")).orderBy(desc("revenue")).limit(5)
top_5_products.show()

+----------+-----------------+
|product_id|          revenue|
+----------+-----------------+
|     P0416|6435762.229999999|
|     P0431|6269029.679999999|
|     P0422|       6250873.32|
|     P0744|       6182377.31|
|     P0267|6169088.670000001|
+----------+-----------------+



In [7]:
high_value_customers = df_cleaned.groupBy("customer_id").agg(sum("total_value").alias("total_spend")).filter(col("total_spend") > 50000)
high_value_customers.show()

+-----------+------------------+
|customer_id|       total_spend|
+-----------+------------------+
|     C29954|135639.24000000002|
|     C17272|122354.37000000001|
|     C30105|         104659.45|
|     C15116|106321.31999999999|
|     C48347|         100666.14|
|     C02501|          115218.0|
|     C02472|         102674.68|
|     C03294|         118237.46|
|     C31344|137165.74999999997|
|     C02165|127707.64000000001|
|     C48177|         123040.87|
|     C23161|         139509.05|
|     C38250|          80693.96|
|     C07741|109399.13999999998|
|     C20715|207963.91000000003|
|     C33879|104327.71999999999|
|     C12865|122521.29999999999|
|     C23223|140742.50999999998|
|     C16587|         115189.26|
|     C23090|113084.60999999999|
+-----------+------------------+
only showing top 20 rows


In [8]:
df_cleaned.createOrReplaceTempView("sales_transactions")

In [9]:
spark.sql("""
    SELECT store_id, date, SUM(total_value) AS total_sales
    FROM sales_transactions
    GROUP BY store_id, date
""").show()

+--------+----------+------------------+
|store_id|      date|       total_sales|
+--------+----------+------------------+
|    S009|2026-03-09|         3817531.9|
|    S032|2026-03-14|3875893.5100000002|
|    S033|2026-03-12|3710183.9200000004|
|    S031|2026-03-11|3812412.1300000004|
|    S031|2026-03-16|3848062.2899999996|
|    S043|2026-03-12|        4119570.06|
|    S021|2026-03-15|3514283.0000000005|
|    S043|2026-03-07|        3334591.48|
|    S043|2026-03-16|3796301.8200000003|
|    S028|2026-04-03|         490172.16|
|    S022|2026-03-08|        4074496.55|
|    S005|2026-03-19|        3697788.13|
|    S019|2026-03-21|3634687.6400000006|
|    S001|2026-03-23|3381885.7800000003|
|    S038|2026-03-04|3167357.6800000006|
|    S029|2026-03-21|        3846533.29|
|    S034|2026-04-03|         489748.05|
|    S003|2026-03-28|3679466.7399999998|
|    S015|2026-03-21|3480435.0600000005|
|    S023|2026-03-20|3812980.4499999997|
+--------+----------+------------------+
only showing top

In [10]:
spark.sql("""
    SELECT product_id, SUM(total_value) AS revenue
    FROM sales_transactions
    GROUP BY product_id
    ORDER BY revenue DESC
    LIMIT 5
""").show()

+----------+-----------------+
|product_id|          revenue|
+----------+-----------------+
|     P0416|6435762.229999999|
|     P0431|6269029.679999999|
|     P0422|       6250873.32|
|     P0744|       6182377.31|
|     P0267|6169088.670000001|
+----------+-----------------+



In [11]:
spark.sql("""
    SELECT customer_id, SUM(total_value) AS total_spend
    FROM sales_transactions
    GROUP BY customer_id
    HAVING SUM(total_value) > 50000
""").show()

+-----------+------------------+
|customer_id|       total_spend|
+-----------+------------------+
|     C29954|135639.24000000002|
|     C17272|122354.37000000001|
|     C30105|         104659.45|
|     C15116|106321.31999999999|
|     C48347|         100666.14|
|     C02501|          115218.0|
|     C02472|         102674.68|
|     C03294|         118237.46|
|     C31344|137165.74999999997|
|     C02165|127707.64000000001|
|     C48177|         123040.87|
|     C23161|         139509.05|
|     C38250|          80693.96|
|     C07741|109399.13999999998|
|     C20715|207963.91000000003|
|     C33879|104327.71999999999|
|     C12865|122521.29999999999|
|     C23223|140742.50999999998|
|     C16587|         115189.26|
|     C23090|113084.60999999999|
+-----------+------------------+
only showing top 20 rows
